In [ ]:
%load_ext autoreload
%autoreload 2
import os
from dotenv import load_dotenv
from TextMiningBasedSatdDetectorModel import TextMiningBasedSatdDetectorModel
from SimpleOutputLabelConverter import SimpleOutputLabelConverter
from constant import *


In [ ]:
load_dotenv()
simple_output_label_converter = SimpleOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)
BASE_SATD_DETECTOR_DIRECTORY = os.getenv('BASE_SATD_DETECTOR_DIRECTORY')

# Training Dataset Preparation

In [ ]:
default_directory = f"{BASE_SATD_DETECTOR_DIRECTORY}/models/default"
os.makedirs(default_directory, exist_ok=True)
os.makedirs(os.path.join(default_directory, 'models'), exist_ok=True)
detect_train_df['text'].str.replace('\n', '\t').to_csv(f'{default_directory}/comments.txt', index=False, header=False)
detect_train_df['label'].str.lower().map({'yes': 'Yes', 'no': 'No'}).to_csv(f'{default_directory}/labels.txt', index=False, header=False)
detect_train_df.assign(project='train')['project'].to_csv(f'{default_directory}/projects.txt', index=False, header=False)

In [ ]:
pretrained_satd_detector = TextMiningBasedSatdDetectorModel('detect', 'pretrained-liu-detector', simple_output_label_converter, default_directory)
pretrained_satd_detector.fit(detect_train_dataset)
pretrained_satd_detector.predict(detect_test_dataset, DATASET_NAME)

In [ ]:
trained_detector = TextMiningBasedSatdDetectorModel('detect', 'trained-liu-detector',simple_output_label_converter, default_directory, retrain=True)
trained_detector.fit(detect_train_dataset)
trained_detector.predict(detect_test_dataset, DATASET_NAME)

5-Fold Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd
df = pd.concat([detect_train_df, detect_test_df])

X = df["text"]
y = df["label"]
groups = df["repository"]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
 X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
 y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
 print(f'Fold {fold + 1}: {len(X_train)} train and {len(X_test)} samples')

 fold_suffix = f"5fcv-{fold+1}"
 fold_directory = f"{BASE_SATD_DETECTOR_DIRECTORY}/models/{fold_suffix}"
 os.makedirs(fold_directory, exist_ok=True)
 os.makedirs(os.path.join(fold_directory, 'models'), exist_ok=True)
 detect_train_df['text'].str.replace('\n', '\t').to_csv(f'{fold_directory}/comments.txt', index=False, header=False)
 detect_train_df['label'].str.lower().map({'yes': 'Yes', 'no': 'No'}).to_csv(f'{fold_directory}/labels.txt', index=False, header=False)
 detect_train_df.assign(project='train')['project'].to_csv(f'{fold_directory}/projects.txt', index=False, header=False)

 trained_detector = TextMiningBasedSatdDetectorModel('detect', f'trained-liu-detector-{fold_suffix}',simple_output_label_converter, fold_directory, retrain=True)
 trained_detector.fit(detect_train_dataset)
 trained_detector.predict(detect_test_dataset, DATASET_NAME)
